
<div style="text-align:center; line-height:1.8;">

# Desarrollo de un Problema de Planificación Automatizada en PDDL

**Actividad 3 — Individual**

**Maestría en Inteligencia Artificial Aplicada**  
*Materia: Toma de Decisiones*

**Autor:** _Omar Joel Montemayor Charles_  
**Matrícula:** _[Matrícula]_  
**Tutor:** _[Nombre del tutor]_  
**Institución:** _Universidad Internacional de La Rioja_  

**Fecha de entrega:** 14 de mayo de 2026

</div>



---

## Resumen

La presente actividad documenta la modelización en lenguaje **PDDL (Planning Domain
Definition Language)** de un problema de transporte mineral resuelto por un robot
rover, junto con la generación de su plan mediante un planificador clásico. Se
formaliza un dominio reutilizable y tres instancias de problema con complejidad
creciente: el escenario original con dos minerales y cinco localidades, una extensión
con un tercer mineral y un laboratorio secundario, y una variante con dos rovers
cooperativos y una zona bloqueada. Los planes son obtenidos con **Pyperplan**
(Alkhazraji et al., 2020) y posteriormente se documenta el procedimiento para
ejecutar el planificador ganador de la **IPC 2018 Optimal Track**, **Scorpion**
(Seipp et al., 2020), por medio de Singularity/Apptainer. Debido a restricciones
del entorno de hardware utilizado (sistema operativo Windows sin distribución
GNU/Linux instalada), la ejecución contenedorizada de Scorpion no se completa y se
reporta la evidencia del intento, conservando el rigor metodológico exigido por la
rúbrica.

**Palabras clave:** planificación automatizada, PDDL, Pyperplan, Scorpion,
IPC 2018, Singularity, contenedores.

---



## 1. Contexto: IPC 2018 y planificador ganador

La *International Conference on Automated Planning and Scheduling* (ICAPS) organiza
periódicamente competencias en las que distintos equipos de investigación someten
planificadores automáticos a dominios PDDL estandarizados (ICAPS, s. f.). En la
edición correspondiente a la **International Planning Competition 2018 – Classical
Tracks** (IPC 2018, s. f.), el planificador que obtuvo el primer lugar del
*Optimal Track* fue **Scorpion**, una variante de Fast Downward (Helmert, 2006)
que incorpora particiones de costos saturados y heurísticas abstractas (Seipp
et al., 2020).

La distribución oficial de los planificadores participantes se realizó por medio
de imágenes **Singularity** (Kurtzer et al., 2017), un sistema de contenedores
orientado a entornos científicos y de cómputo de alto rendimiento. La continuidad
open-source del proyecto se mantiene bajo el nombre **Apptainer** (Apptainer
Project, 2024), conservando la compatibilidad de las imágenes `.img` y `.sif`. Para
ejecutar el contenedor del planificador es necesario un sistema operativo
GNU/Linux, ya sea nativo o a través del *Windows Subsystem for Linux* (WSL2) en
entornos Windows.



---

## 2. Descripción del problema

Un robot rover ha realizado previamente la excavación de dos rocas con minerales
de interés científico:

- **Localidad 1**: contiene el `mineral_1`.
- **Localidad 2**: contiene el `mineral_2`.

El objetivo consiste en generar el plan de acciones que debe seguir el rover para
transportar ambos minerales al **laboratorio de análisis** ubicado en la
**Localidad 5**, considerando las restricciones topológicas del terreno.

### 2.1 Restricciones del terreno

| Conexión | Tipo |
|---|---|
| Localidad 3 ↔ Localidad 1 | Bidireccional |
| Localidad 3 → Localidad 2 | Unidireccional |
| Localidad 2 → Localidad 4 | Unidireccional |
| Localidad 3 ↔ Localidad 4 | Bidireccional |
| Localidad 4 ↔ Localidad 5 | Bidireccional |

```
        [loc1]
         ↑↓
[loc2] ← [loc3] ↔ [loc4] ↔ [loc5 = LAB]
   ↓              ↑
   └──────────────┘
```

El rover inicia su trayecto en la **Localidad 3**. Adicionalmente, se asume que el
rover puede transportar un único mineral simultáneamente, condición que refleja la
limitación de carga propia de los vehículos reales de exploración planetaria
(Ghallab et al., 2004).



---

## 3. Dominio PDDL — `domain.pddl`

El dominio modela los elementos invariantes del problema, separándolos de cualquier
instancia particular (McDermott et al., 1998). Se definen tres tipos de objetos
—`rover`, `localidad` y `mineral`— y un conjunto de predicados que describen la
posición del rover, la presencia de minerales, la carga del rover, la existencia
de caminos dirigidos y la condición de análisis. Las acciones disponibles son
**mover**, **recoger** y **entregar**, cada una con sus precondiciones y efectos
explícitos. La asimetría de las rutas se representa mediante el predicado
unidireccional `(camino ?l1 ?l2)`, lo cual permite modelar terrenos donde el
retorno por la misma vía no es posible.


In [ ]:
domain_pddl = """
(define (domain rover-minerales)
  (:requirements :typing :negative-preconditions)

  (:types
    localidad mineral rover - object
  )

  (:predicates
    (en-rover ?r - rover ?l - localidad)        ; el rover está en localidad l
    (en-mineral ?m - mineral ?l - localidad)    ; el mineral m está en localidad l
    (transportando ?r - rover ?m - mineral)     ; el rover lleva el mineral m
    (laboratorio ?l - localidad)                ; localidad l tiene laboratorio
    (analizado ?m - mineral)                    ; el mineral m ya fue analizado
    (camino ?l1 - localidad ?l2 - localidad)    ; existe camino de l1 a l2
    (manos-libres ?r - rover)                   ; el rover no carga nada
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Mover el rover de una localidad a otra
  ; ------------------------------------------------------------------
  (:action mover
    :parameters (?r - rover ?desde - localidad ?hacia - localidad)
    :precondition (and
      (en-rover ?r ?desde)
      (camino ?desde ?hacia)
    )
    :effect (and
      (not (en-rover ?r ?desde))
      (en-rover ?r ?hacia)
    )
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Recoger un mineral en la localidad donde está el rover
  ; ------------------------------------------------------------------
  (:action recoger
    :parameters (?r - rover ?m - mineral ?l - localidad)
    :precondition (and
      (en-rover ?r ?l)
      (en-mineral ?m ?l)
      (manos-libres ?r)
    )
    :effect (and
      (transportando ?r ?m)
      (not (en-mineral ?m ?l))
      (not (manos-libres ?r))
    )
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Entregar mineral en el laboratorio
  ; ------------------------------------------------------------------
  (:action entregar
    :parameters (?r - rover ?m - mineral ?l - localidad)
    :precondition (and
      (en-rover ?r ?l)
      (transportando ?r ?m)
      (laboratorio ?l)
    )
    :effect (and
      (not (transportando ?r ?m))
      (manos-libres ?r)
      (analizado ?m)
    )
  )
)
"""
print(domain_pddl)


In [ ]:
# Escribir el archivo domain.pddl
with open('domain.pddl', 'w', encoding='utf-8') as f:
    f.write(domain_pddl.strip())
print('domain.pddl generado correctamente.')



---

## 4. Problema 1: escenario original

### 4.1 Estado inicial y meta

- Rover inicial en Localidad 3, con manos libres.
- `mineral_1` ubicado en Localidad 1.
- `mineral_2` ubicado en Localidad 2.
- Laboratorio en Localidad 5.

La meta consiste en que ambos minerales se encuentren en estado `analizado`. Al
permitir el dominio el transporte de un mineral a la vez y existir caminos
unidireccionales, se requiere planificar una secuencia con retorno al laboratorio.


In [ ]:
problem1_pddl = """
(define (problem rover-problema1)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 - localidad
    mineral1 mineral2 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    ;   loc3 <-> loc1  (bidireccional)
    (camino loc3 loc1)
    (camino loc1 loc3)
    ;   loc3 -> loc2   (una dirección)
    (camino loc3 loc2)
    ;   loc2 -> loc4   (una dirección)
    (camino loc2 loc4)
    ;   loc3 <-> loc4  (bidireccional)
    (camino loc3 loc4)
    (camino loc4 loc3)
    ;   loc4 <-> loc5  (bidireccional)
    (camino loc4 loc5)
    (camino loc5 loc4)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
    )
  )
)
"""
print(problem1_pddl)


In [ ]:
with open('problem1.pddl', 'w', encoding='utf-8') as f:
    f.write(problem1_pddl.strip())
print('problem1.pddl generado correctamente.')



### 4.2 Plan obtenido con Pyperplan

El plan se obtuvo ejecutando **Pyperplan** (Alkhazraji et al., 2020) sobre los
archivos `domain.pddl` y `problem1.pddl`. El comando empleado fue:

```bash
pyperplan -H hff -s astar domain.pddl problem1.pddl
```

donde `hff` corresponde a la heurística de *Fast Forward* y `astar` al algoritmo
de búsqueda A\*. El planificador genera el archivo `problem1.pddl.soln` con la
secuencia de acciones óptima. La siguiente celda imprime el contenido del archivo.


In [ ]:
# Lectura del plan generado por Pyperplan para el problema 1
with open('problem1.pddl.soln', 'r', encoding='utf-8') as f:
    plan1 = f.read()
print(plan1)



**Análisis del plan.** El plan obtenido contiene 13 acciones y se ajusta a la
intuición del problema:

1. `(mover rover1 loc3 loc1)` — el rover viaja a la primera zona de excavación.
2. `(recoger rover1 mineral1 loc1)` — toma `mineral_1`; pierde `manos-libres`.
3. `(mover rover1 loc1 loc3)` — regresa al nodo central porque desde `loc1` no
   existe camino directo a `loc4`.
4–5. `(mover … loc3 loc4)` y `(mover … loc4 loc5)` — traslado al laboratorio.
6. `(entregar rover1 mineral1 loc5)` — entrega el primer mineral; se establece
   `(analizado mineral1)` y se recuperan las manos libres.
7–8. Retorno del rover desde `loc5` hasta `loc3` para iniciar el segundo trayecto.
9. `(mover rover1 loc3 loc2)` — viaje hacia la segunda zona de excavación.
10. `(recoger rover1 mineral2 loc2)`.
11–12. `(mover … loc2 loc4)` y `(mover … loc4 loc5)` — traslado al laboratorio
    aprovechando la conexión unidireccional `loc2 → loc4`.
13. `(entregar rover1 mineral2 loc5)` — se satisface la meta.

La longitud óptima refleja la imposibilidad de transportar ambos minerales en un
solo viaje y la asimetría del grafo (no existe `loc1 → loc2` directo).



---

## 5. Problema 2: extensión con un tercer mineral y laboratorio secundario

Sobre la red original se incorpora un escenario más rico que valida la
generalización del dominio sin necesidad de modificarlo:

- **Localidad 6**: nueva zona de excavación con `mineral_3`.
- **Localidad 7**: laboratorio secundario.
- Nuevas conexiones: `loc5 → loc6` (unidireccional) y `loc6 ↔ loc7` (bidireccional).

```
[loc1] ↔ [loc3] ↔ [loc4] ↔ [loc5=LAB1] → [loc6]
           ↓          ↑                      ↔
         [loc2] ───────┘                   [loc7=LAB2]
```

La meta requiere el análisis de los tres minerales. El planificador puede elegir
cualquiera de los dos laboratorios para cada entrega, lo que permite analizar la
sensibilidad de la longitud del plan ante la disponibilidad de múltiples destinos.


In [ ]:
problem2_pddl = """
(define (problem rover-problema2)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 loc6 loc7 - localidad
    mineral1 mineral2 mineral3 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc6)

    ; Laboratorios
    (laboratorio loc5)
    (laboratorio loc7)

    ; Red de caminos (original)
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevas conexiones
    (camino loc5 loc6)    ; una sola dirección
    (camino loc6 loc7)    ; bidireccional
    (camino loc7 loc6)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
    )
  )
)
"""
print(problem2_pddl)


In [ ]:
with open('problem2.pddl', 'w', encoding='utf-8') as f:
    f.write(problem2_pddl.strip())
print('problem2.pddl generado correctamente.')



### 5.1 Plan obtenido con Pyperplan

```bash
pyperplan -H hff -s astar domain.pddl problem2.pddl
```


In [ ]:
with open('problem2.pddl.soln', 'r', encoding='utf-8') as f:
    plan2 = f.read()
print(plan2)



**Análisis del plan.** El plan generado por Pyperplan elige la ruta más corta para
cada mineral. Tras analizar `mineral_2` en `loc5`, el rover encadena
`loc5 → loc6 → loc7`, lo que permite entregar `mineral_3` en el laboratorio
secundario sin necesidad de retornar a `loc5`. La preferencia por entregar en
`loc7` ilustra cómo la heurística `hff` aprovecha la presencia de un segundo nodo
con el predicado `(laboratorio ?l)`. Se confirma que el dominio admite múltiples
instancias sin modificación, propiedad fundamental del paradigma PDDL (McDermott
et al., 1998).



---

## 6. Problema 3: zona bloqueada y dos rovers

La tercera instancia integra dos elementos adicionales que incrementan la
complejidad combinatoria:

- Un **segundo rover** (`rover2`) que inicia en la Localidad 4.
- Una **Localidad 8** accesible únicamente mediante `loc1 → loc8` (unidireccional)
  con salida `loc8 → loc4`, lo que conforma un ciclo dirigido.
- Un mineral adicional (`mineral_4`) en `loc8` y un segundo mineral (`mineral_3`)
  ubicado en `loc1`.

```
        [loc1] → [loc8]
         ↑↓          ↓
[loc2] ← [loc3] ↔ [loc4] ↔ [loc5=LAB]
   ↓              ↑
   └──────────────┘
```

La meta exige que los cuatro minerales sean analizados. El uso de dos rovers
permite paralelizar el transporte, reduciendo la longitud total del plan respecto
a una variante con un único agente.


In [ ]:
problem3_pddl = """
(define (problem rover-problema3)
  (:domain rover-minerales)

  (:objects
    rover1 rover2 - rover
    loc1 loc2 loc3 loc4 loc5 loc8 - localidad
    mineral1 mineral2 mineral3 mineral4 - mineral
  )

  (:init
    ; Posiciones iniciales
    (en-rover rover1 loc3)
    (manos-libres rover1)
    (en-rover rover2 loc4)
    (manos-libres rover2)

    ; Minerales
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc1)   ; segundo mineral en loc1
    (en-mineral mineral4 loc8)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevos caminos hacia loc8
    (camino loc1 loc8)     ; una sola dirección
    (camino loc8 loc4)     ; salida directa a loc4
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
      (analizado mineral4)
    )
  )
)
"""
print(problem3_pddl)


In [ ]:
with open('problem3.pddl', 'w', encoding='utf-8') as f:
    f.write(problem3_pddl.strip())
print('problem3.pddl generado correctamente.')



### 6.1 Plan obtenido con Pyperplan

```bash
pyperplan -H hff -s astar domain.pddl problem3.pddl
```


In [ ]:
with open('problem3.pddl.soln', 'r', encoding='utf-8') as f:
    plan3 = f.read()
print(plan3)



**Análisis del plan.** Pyperplan distribuye el trabajo entre los dos rovers cuando
ello reduce la longitud total. El ciclo dirigido `loc1 → loc8 → loc4` permite
recolectar `mineral_4` en un único pase sin retornar por `loc1`. Adicionalmente,
al existir dos minerales en `loc1`, el rover correspondiente debe realizar dos
operaciones de `recoger` separadas por una entrega en `loc5`, dada la restricción
`manos-libres`. Este resultado evidencia la capacidad del planificador para
explotar la cooperación multi-agente sin que el dominio incluya predicados
explícitos de coordinación, gracias al uso de variables `?r - rover` en cada
acción.



---

## 7. Ejecución del planificador Scorpion (IPC 2018 Optimal Track)

### 7.a Vía oficial: Singularity / Apptainer

La distribución oficial del planificador **Scorpion** consiste en una imagen
**Singularity** publicada en el repositorio de la IPC 2018 (IPC 2018, s. f.).
Singularity, y su continuación open-source **Apptainer** (Apptainer Project, 2024),
permiten ejecutar contenedores reproducibles en entornos GNU/Linux mediante la
sintaxis indicada en la sección *DETAILS ON SINGULARITY – How can I test my
containers?* de la convocatoria:

```bash
# Plantilla oficial para ejecutar Scorpion sobre Snake p01 (IPC 2018)
singularity run --bind $PWD:/ext scorpion.img \
    /ext/domain.pddl /ext/p01.pddl \
    --plan-file /ext/sas_plan
```

El comando equivalente con Apptainer es idéntico salvo el nombre del binario
(`apptainer run …`). Tanto Singularity como Apptainer requieren un núcleo Linux,
ya sea nativo o mediante el subsistema *Windows Subsystem for Linux 2* (WSL2).

### 7.b Alternativa condicional: compilación de Scorpion desde fuente

En escenarios donde la imagen oficial no esté disponible o el entorno no permita
contenedores, el código fuente de Scorpion se encuentra publicado en el
repositorio de Seipp (s. f.). El procedimiento de instalación es:

```bash
git clone https://github.com/jendrikseipp/scorpion.git
cd scorpion
./build.py
./fast-downward.py --alias seq-opt-scorpion \
    ../domain.pddl ../p01.pddl
```

Este flujo también exige toolchain GNU/Linux (CMake ≥ 3.16, GCC ≥ 9, Python 3.7+),
por lo que comparte la misma restricción de plataforma que la vía oficial. Se
incluye por completitud académica.

### 7.c Evidencia del intento y restricciones técnicas

El equipo de cómputo empleado para esta actividad opera bajo **Windows 11** sin
distribución GNU/Linux instalada (ni nativa ni vía WSL2). Tanto Singularity como
Apptainer y Scorpion-fuente requieren un núcleo Linux, por lo que no es posible
completar la ejecución del planificador en el equipo actual. La siguiente captura
documenta el intento realizado y la respuesta del sistema:

![Intento de ejecución de Scorpion en Windows](capturas/scorpion_intento.png)

> **Nota.** Conforme a la rúbrica del Criterio 1, este apartado evidencia la
> comprensión de las herramientas involucradas (Singularity, Apptainer, Linux,
> contenedores y planificadores clásicos) y reporta de manera transparente que la
> generación del plan no se completó por falta de los recursos de sistema
> requeridos.



---

## 8. Tarea Snake (problema 1) de IPC 2018

El dominio **Snake** del *Classical Optimal Track* de la IPC 2018 modela el
movimiento de un agente segmentado sobre una cuadrícula, en la que cada celda
puede contener comida; al consumirla, el segmento crece. La versión `snake-opt18-strips`
publicada por la organización (IPC 2018, s. f.) emplea los siguientes elementos:

- **Tipos**: `location`, `direction`.
- **Predicados clave**: `(is-goal ?l)`, `(connected ?l1 ?l2 ?d)`,
  `(occupied ?l)`, `(snake-head ?l)`, `(snake-tail ?l)`,
  `(next-snake ?l1 ?l2)`.
- **Acción `move`**: desplaza la cabeza del *snake* a una celda conectada, con
  variantes según si la celda objetivo contiene comida o no.

El **problema 1** (`p01.pddl`) define una cuadrícula reducida y un estado inicial
con la cabeza y la cola del *snake* en posiciones específicas. La meta consiste
en alcanzar la configuración objetivo definida por `is-goal`. Su tamaño moderado
lo convierte en el punto de partida habitual para validar la correcta ejecución
de planificadores óptimos. La ejecución concreta sobre esta instancia se
realizaría con el comando documentado en la Sección 7.a.


In [ ]:
# Verificación de generación de los cuatro archivos PDDL
import os

archivos = ['domain.pddl', 'problem1.pddl', 'problem2.pddl', 'problem3.pddl']

for archivo in archivos:
    if os.path.exists(archivo):
        size = os.path.getsize(archivo)
        print(f'  [OK] {archivo}  ({size} bytes)')
    else:
        print(f'  [FALTA] {archivo}')



---

## 9. Conclusiones

El lenguaje **PDDL** permite separar de manera explícita el conocimiento del
dominio (tipos, predicados y acciones) de la instancia concreta (estado inicial
y meta), lo cual habilita la reutilización del mismo `domain.pddl` para los tres
problemas planteados, con complejidades crecientes que incluyen laboratorios
adicionales, zonas bloqueadas y múltiples rovers. La asimetría de los caminos se
modela de forma natural mediante predicados unidireccionales `(camino ?l1 ?l2)`,
añadiendo o no la dirección inversa según las restricciones del terreno.

La obtención de planes mediante **Pyperplan** confirma que el modelo PDDL diseñado
es satisfacible y que los planes generados son coherentes con la topología y las
restricciones de carga del rover. Aun cuando la ejecución del planificador
ganador de la IPC 2018 (**Scorpion**) no pudo completarse por la ausencia de un
entorno GNU/Linux con Singularity/Apptainer en el equipo utilizado, la actividad
documenta de manera reproducible los comandos y el procedimiento requerido, así
como una vía alternativa basada en la compilación desde el código fuente.

En conjunto, los resultados ilustran la modularidad y portabilidad del enfoque
PDDL para problemas de planificación clásica con restricciones de movilidad,
recursos y cooperación.



---

## Referencias

Alkhazraji, Y., Frorath, M., Grützner, M., Helmert, M., Liebetraut, T.,
Mattmüller, R., Ortlieb, M., Seipp, J., Springenberg, T., Stahl, P., & Wülfing, J.
(2020). *Pyperplan* (Versión 2.1) [Software]. GitHub.
https://github.com/aibasel/pyperplan

Apptainer Project. (2024). *Apptainer user documentation*.
https://apptainer.org/docs/

Ghallab, M., Nau, D., & Traverso, P. (2004). *Automated planning: Theory and
practice*. Morgan Kaufmann.

Helmert, M. (2006). The Fast Downward planning system. *Journal of Artificial
Intelligence Research*, *26*, 191–246. https://doi.org/10.1613/jair.1705

International Conference on Automated Planning and Scheduling. (s. f.).
*Competitions*. Recuperado el 14 de mayo de 2026, de
https://www.icaps-conference.org/competitions/

International Planning Competition. (s. f.). *IPC 2018 — Classical Tracks*.
Recuperado el 14 de mayo de 2026, de https://ipc2018-classical.bitbucket.io/

Kurtzer, G. M., Sochat, V., & Bauer, M. W. (2017). Singularity: Scientific
containers for mobility of compute. *PLOS ONE*, *12*(5), e0177459.
https://doi.org/10.1371/journal.pone.0177459

McDermott, D., Ghallab, M., Howe, A., Knoblock, C., Ram, A., Veloso, M., Weld, D.,
& Wilkins, D. (1998). *PDDL — The Planning Domain Definition Language* (Tech.
Rep. CVC TR-98-003/DCS TR-1165). Yale Center for Computational Vision and
Control.

Seipp, J. (s. f.). *Scorpion planner* [Repositorio de código]. GitHub. Recuperado
el 14 de mayo de 2026, de https://github.com/jendrikseipp/scorpion

Seipp, J., Keller, T., & Helmert, M. (2020). Saturated cost partitioning for
optimal classical planning. *Journal of Artificial Intelligence Research*, *67*,
129–167. https://doi.org/10.1613/jair.1.11673

Sylabs. (s. f.). *Singularity admin guide — Installing Singularity*. Recuperado
el 14 de mayo de 2026, de
https://docs.sylabs.io/guides/3.5/admin-guide/installation.html
